# NORTHSTAR -- Tower 19: Windows QA for Solace Browser

**Tower:** 19 -- Tower of Windows
**DNA:** `windows(native) = msi(signed) x defender(trusted) x registry(clean) x hidpi(scaled) x narrator(accessible) x enterprise(gpo) x wsl(compatible)`
**Target:** solace-browser at `/home/phuc/projects/solace-browser/`
**Floors probed:** F1 (MSI/NSIS Installer), F2 (Code Signing / Defender), F3 (Fluent Design / Windows UI), F4 (High DPI Scaling), F5 (Windows Credential Storage), F6 (Taskbar / System Tray), F7 (Toast Notifications), F8 (PowerShell / Path Handling), F9 (Windows Service / Scheduled Task), F10 (Registry Integration)
**Protocol:** FORECAST -> CROSS-REF -> AUDIT -> FIX -> VERIFY -> SEAL -> POSTMORTEM
**Total Questions:** 343 (49 floors x 7 questions)
**Auth:** 65537

**Verdict threshold:** 30% (early-stage Windows port)

In [ ]:
# Cell 1: Bootstrap -- imports, paths, helper functions
# Tower 19: Windows QA -- probing solace-browser for Windows platform readiness
import os, sys, re, json, hashlib, glob as globmod
from pathlib import Path
from datetime import datetime

NORTHSTAR = "windows-qa"
NOTEBOOK_PATH = "notebooks/qa/10-windows-qa.ipynb"
TOWER = 19
TOWER_NAME = "Tower of Windows"
MIN_SCORE = 30  # early-stage Windows port

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
assert BROWSER_ROOT.exists(), f"FATAL: BROWSER_ROOT {BROWSER_ROOT} does not exist"

WEB = BROWSER_ROOT / "web"
SRC = BROWSER_ROOT / "src"
SCRIPTS_SRC = SRC / "scripts"
SCRIPTS_TOP = BROWSER_ROOT / "scripts"
DATA = BROWSER_ROOT / "data" / "default"
CSS_FILE = WEB / "css" / "site.css"
JS_DIR = WEB / "js"
RESOURCES = BROWSER_ROOT / "resources"

results = []

def record(probe_id, name, passed, detail=""):
    """Record a PASS/FAIL result with evidence detail."""
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    """Safely read a file, returning empty string if missing."""
    p = Path(path)
    return p.read_text(encoding="utf-8", errors="replace") if p.exists() else ""

def file_exists(path):
    return Path(path).exists()

def grep_files(root, pattern, extensions=None):
    """Search files under root for regex pattern. Returns {relpath: [matching_lines]}."""
    if extensions is None:
        extensions = [".py", ".js", ".sh", ".json", ".toml", ".spec",
                      ".html", ".css", ".yml", ".yaml", ".md", ".cfg",
                      ".ps1", ".bat", ".cmd"]
    hits = {}
    for ext in extensions:
        for fpath in Path(root).rglob(f"*{ext}"):
            if any(skip in fpath.parts for skip in (".git", "node_modules", ".venv", "dist")):
                continue
            try:
                text = fpath.read_text(errors="ignore")
                for i, line in enumerate(text.splitlines(), 1):
                    if re.search(pattern, line, re.IGNORECASE):
                        rel = str(fpath.relative_to(BROWSER_ROOT))
                        hits.setdefault(rel, []).append(f"L{i}: {line.strip()[:120]}")
            except Exception:
                pass
    return hits

# Read build scripts and spec files for reuse across probes
build_win_src = read_file(SCRIPTS_SRC / "build-windows.sh")
build_win_top = read_file(SCRIPTS_TOP / "build-windows.sh")
combined_build = build_win_src + build_win_top
generic_spec = read_file(BROWSER_ROOT / "solace-browser.spec")

print(f"Bootstrap OK -- BROWSER_ROOT: {BROWSER_ROOT}")
print(f"Timestamp: {datetime.now().isoformat()}")

In [ ]:
# ============================================================
# Floor 1: MSI/NSIS Installer
# Persona: MSI Installer Expert (Tower 19, Floor 1)
# Questions probed:
#   Q1: Do build scripts reference MSI creation (WiX, Tauri, Inno Setup)?
#   Q2: Is there an NSIS or Inno Setup script?
#   Q3: Does the installer package have a versioned name?
#   Q4: Are SHA-256 checksums generated for the installer?
# ============================================================
print("=" * 60)
print("FLOOR 1: MSI/NSIS Installer")
print("=" * 60)

# Q1: MSI/installer references
has_msi = bool(re.search(r"\.msi|WiX|wix|light\.exe|candle\.exe|tauri.*build.*windows", combined_build, re.IGNORECASE))
has_inno = bool(re.search(r"inno.setup|iscc|InnoSetup|package-windows-installer", combined_build, re.IGNORECASE))
record("F1-Q1", "MSI or Inno Setup in build scripts", has_msi or has_inno,
       f"MSI: {has_msi}, Inno Setup: {has_inno}")

# Q2: NSIS/Inno script files
nsis_files = grep_files(BROWSER_ROOT, r"nsis|inno.setup|iscc|\[Setup\]|\[Files\]",
                        extensions=[".nsi", ".iss", ".ps1", ".sh"])
record("F1-Q2", "NSIS/Inno Setup script files", len(nsis_files) > 0,
       f"{len(nsis_files)} files with installer script references")
for fp, lines in list(nsis_files.items())[:3]:
    print(f"  {fp}: {lines[0]}")

# Q3: Versioned installer naming
installer_name = re.search(r"SolaceBrowser-.*\.(msi|exe)", combined_build)
record("F1-Q3", "Versioned installer naming", installer_name is not None,
       f"Pattern: {installer_name.group(0)}" if installer_name
       else "No versioned installer pattern found")

# Q4: SHA-256 checksums
has_checksum = bool(re.search(r"sha256sum|sha256|hashlib\.sha256", combined_build, re.IGNORECASE))
record("F1-Q4", "SHA-256 checksums for installer", has_checksum,
       "Checksum generation found" if has_checksum
       else "No checksum generation in Windows build script")

In [ ]:
# ============================================================
# Floor 2: Windows Defender / SmartScreen Compatibility
# Persona: Windows Defender Compatibility Tester (Tower 19, Floor 2)
# Questions probed:
#   Q1: Is there code signing for the Windows executable?
#   Q2: Does the generic .spec reference a .ico icon?
#   Q3: Is there a version_info.txt for the Windows binary?
#   Q4: Does the build script reference signtool or code signing?
# ============================================================
print("=" * 60)
print("FLOOR 2: Windows Defender / SmartScreen")
print("=" * 60)

# Q1: Code signing references
sign_hits = grep_files(BROWSER_ROOT, r"signtool|authenticode|code.?sign|ev.?cert|smartscreen",
                       extensions=[".sh", ".ps1", ".bat", ".yml", ".yaml"])
record("F2-Q1", "Code signing for Windows", len(sign_hits) > 0,
       f"{len(sign_hits)} files reference code signing" if sign_hits
       else "No signtool/authenticode references found")

# Q2: .ico in generic spec
has_ico = bool(re.search(r"icon.*=.*\.ico|\.ico", generic_spec, re.IGNORECASE))
record("F2-Q2", ".ico icon in generic spec", has_ico,
       "Windows .ico icon referenced in solace-browser.spec" if has_ico
       else "No .ico reference in spec")

# Q3: version_info.txt
has_ver_info = bool(re.search(r"version.*info|version_info", generic_spec, re.IGNORECASE))
ver_info_path = RESOURCES / "windows" / "version_info.txt"
ver_info_exists = ver_info_path.exists()
record("F2-Q3", "version_info.txt for Windows",
       has_ver_info or ver_info_exists,
       f"Spec ref: {has_ver_info}, File exists: {ver_info_exists}")

# Q4: signtool in Windows build
has_signtool = bool(re.search(r"signtool|sign|certificate", combined_build, re.IGNORECASE))
record("F2-Q4", "signtool in Windows build script", has_signtool,
       "Signing references found" if has_signtool
       else "No signtool or signing in Windows build")

In [ ]:
# ============================================================
# Floor 3: Fluent Design / Windows UI Patterns
# Persona: Windows UX Designer (Tower 19, Floor 3)
# Questions probed:
#   Q1: Does the CSS support prefers-color-scheme (dark/light)?
#   Q2: Are standard Windows shortcuts (Ctrl+C/V/X, Alt+F4) handled?
#   Q3: Is there Fluent-inspired styling (rounded corners, acrylic)?
#   Q4: Does the CSS handle Windows-specific font rendering?
# ============================================================
print("=" * 60)
print("FLOOR 3: Fluent Design / Windows UI")
print("=" * 60)

css_content = read_file(CSS_FILE)

# Q1: prefers-color-scheme
has_color_scheme = bool(re.search(r"prefers-color-scheme", css_content))
record("F3-Q1", "prefers-color-scheme in CSS", has_color_scheme,
       "Dark/light mode CSS support" if has_color_scheme
       else "No prefers-color-scheme media query")

# Q2: Standard Windows shortcuts in JS
all_js = ""
if JS_DIR.exists():
    for f in JS_DIR.rglob("*.js"):
        try:
            all_js += f.read_text(errors="ignore")
        except Exception:
            pass

has_ctrl = bool(re.search(r"ctrlKey|Ctrl|Control", all_js))
has_alt_f4 = bool(re.search(r"Alt.*F4|AltF4|alt.*f4", all_js))
record("F3-Q2", "Standard Windows shortcuts (Ctrl+, Alt+F4)",
       has_ctrl,
       f"ctrlKey: {has_ctrl}, Alt+F4: {has_alt_f4}")

# Q3: Fluent-inspired styling
fluent_patterns = r"border-radius|backdrop-filter|blur\(|acrylic|mica|rounded"
has_fluent = bool(re.search(fluent_patterns, css_content, re.IGNORECASE))
record("F3-Q3", "Fluent-inspired styling (rounded, blur)", has_fluent,
       "Fluent-like CSS found" if has_fluent
       else "No rounded corners, backdrop-filter, or blur in CSS")

# Q4: Windows font rendering (Segoe UI, system-ui)
has_segoe = bool(re.search(r"Segoe.UI|system-ui|-apple-system", css_content))
record("F3-Q4", "Windows-friendly font stack", has_segoe,
       "Segoe UI or system-ui in CSS" if has_segoe
       else "No Windows-specific font stack")

In [ ]:
# ============================================================
# Floor 4: High DPI Scaling
# Persona: HiDPI Display Specialist (Tower 19, Floor 4)
# Questions probed:
#   Q1: Does CSS handle viewport/scaling units properly?
#   Q2: Are there any devicePixelRatio or DPI-aware references?
#   Q3: Are icon files provided at multiple sizes?
#   Q4: Does the build/config set DPI awareness flags?
# ============================================================
print("=" * 60)
print("FLOOR 4: High DPI Scaling")
print("=" * 60)

# Q1: Responsive CSS units (rem, em, vw, vh) vs fixed px
has_responsive = bool(re.search(r"\d+rem|\d+em|\d+vw|\d+vh", css_content))
record("F4-Q1", "Responsive CSS units (rem/em/vw/vh)", has_responsive,
       "Responsive units found in CSS" if has_responsive
       else "Only fixed px units in CSS")

# Q2: devicePixelRatio or DPI references
dpi_js = grep_files(JS_DIR if JS_DIR.exists() else WEB,
                    r"devicePixelRatio|dpi|scaling|hidpi",
                    extensions=[".js"])
dpi_py = grep_files(SRC, r"dpi|scaling|hidpi|high.?resolution",
                    extensions=[".py"])
record("F4-Q2", "DPI-aware code (devicePixelRatio)",
       len(dpi_js) > 0 or len(dpi_py) > 0,
       f"JS refs: {len(dpi_js)}, Python refs: {len(dpi_py)}")

# Q3: Multiple icon sizes
windows_res = RESOURCES / "windows" if RESOURCES.exists() else None
icon_files = []
if windows_res and windows_res.exists():
    icon_files = [f.name for f in windows_res.iterdir() if f.suffix in (".ico", ".png")]
record("F4-Q3", "Multiple icon sizes in resources/windows",
       len(icon_files) > 0,
       f"Icons: {icon_files[:5]}" if icon_files
       else "resources/windows has no icon files")

# Q4: DPI awareness in build config
dpi_flags = grep_files(BROWSER_ROOT, r"dpi.?aware|SetProcessDPIAware|manifest.*dpi",
                       extensions=[".py", ".json", ".xml", ".manifest"])
record("F4-Q4", "DPI awareness flags in config", len(dpi_flags) > 0,
       f"{len(dpi_flags)} DPI awareness references" if dpi_flags
       else "No DPI awareness manifest or flags")

In [ ]:
# ============================================================
# Floor 5: Windows Credential Storage
# Persona: Windows Credential Manager Expert (Tower 19, Floor 7)
# Questions probed:
#   Q1: Does any code reference Windows Credential Manager?
#   Q2: Is there keytar, wincred, or DPAPI usage?
#   Q3: Does the vault have platform-aware credential storage?
#   Q4: Is AES-256-GCM encryption used for the local vault?
# ============================================================
print("=" * 60)
print("FLOOR 5: Windows Credential Storage")
print("=" * 60)

# Q1: Windows Credential Manager
cred_hits = grep_files(BROWSER_ROOT,
    r"credential.manager|CredentialManager|wincred|DPAPI|CryptProtectData",
    extensions=[".py", ".js", ".rs"])
record("F5-Q1", "Windows Credential Manager references", len(cred_hits) > 0,
       f"{len(cred_hits)} files" if cred_hits
       else "No Windows Credential Manager references")

# Q2: keytar or platform credential modules
keytar_hits = grep_files(BROWSER_ROOT,
    r"keytar|node-keytar|wincredential|windows.?hello",
    extensions=[".py", ".js", ".json"])
record("F5-Q2", "keytar or wincred module", len(keytar_hits) > 0,
       f"{len(keytar_hits)} files" if keytar_hits
       else "No keytar or wincred module references")

# Q3: Platform-aware credential storage in vault code
vault_hits = grep_files(BROWSER_ROOT,
    r"vault|credential.?store|token.?store|secret.?store",
    extensions=[".py", ".js"])
record("F5-Q3", "Vault/credential store code exists", len(vault_hits) > 0,
       f"{len(vault_hits)} files with vault/credential store references")

# Q4: AES-256-GCM encryption
aes_hits = grep_files(BROWSER_ROOT, r"AES.*GCM|aes.256.gcm|AESGCM|cryptography",
                      extensions=[".py", ".js"])
record("F5-Q4", "AES-256-GCM encryption for vault", len(aes_hits) > 0,
       f"{len(aes_hits)} files with AES-GCM references" if aes_hits
       else "No AES-256-GCM references found")

In [ ]:
# ============================================================
# Floor 6: Taskbar / System Tray Integration
# Persona: Taskbar Integration Expert (Tower 19, Floor 8)
# Questions probed:
#   Q1: Does code reference system tray / notification area?
#   Q2: Is there a Jump List or taskbar progress reference?
#   Q3: Does the .ico icon exist for taskbar display?
#   Q4: Is there tray-based background operation?
# ============================================================
print("=" * 60)
print("FLOOR 6: Taskbar / System Tray")
print("=" * 60)

# Q1: System tray references
tray_hits = grep_files(BROWSER_ROOT,
    r"system.?tray|systray|notification.?area|tray.?icon|pystray|AppIndicator",
    extensions=[".py", ".js", ".rs", ".json"])
record("F6-Q1", "System tray / notification area code", len(tray_hits) > 0,
       f"{len(tray_hits)} files with tray references" if tray_hits
       else "No system tray references")

# Q2: Jump List or taskbar progress
jump_hits = grep_files(BROWSER_ROOT,
    r"jump.?list|JumpList|taskbar.?progress|ITaskbarList|setProgressBar",
    extensions=[".py", ".js", ".rs"])
record("F6-Q2", "Jump List or taskbar progress", len(jump_hits) > 0,
       f"{len(jump_hits)} files" if jump_hits
       else "No Jump List or taskbar progress references")

# Q3: .ico file exists
ico_path = RESOURCES / "windows" / "solace-browser.ico"
ico_glob = list(BROWSER_ROOT.rglob("*.ico"))
ico_glob = [f for f in ico_glob
            if not any(s in str(f) for s in (".git", ".venv", "node_modules"))]
record("F6-Q3", ".ico icon file exists", ico_path.exists() or len(ico_glob) > 0,
       f"Expected: {ico_path.exists()}, Total .ico: {len(ico_glob)} "
       f"{[str(f.relative_to(BROWSER_ROOT)) for f in ico_glob[:3]]}")

# Q4: Background/tray operation mode
bg_hits = grep_files(BROWSER_ROOT,
    r"background.?mode|minimize.?to.?tray|hide.?to.?tray|daemon|headless",
    extensions=[".py", ".js"])
record("F6-Q4", "Background / tray operation mode", len(bg_hits) > 0,
       f"{len(bg_hits)} files with background/tray mode references")

In [ ]:
# ============================================================
# Floor 7: Windows Toast Notifications
# Persona: Windows Notification Expert (Tower 19, Floor 9)
# Questions probed:
#   Q1: Does code reference toast notifications or win10toast?
#   Q2: Is there a push notification / alert system?
#   Q3: Does the YinYang system send OS-level notifications?
#   Q4: Are notifications actionable (buttons in toast)?
# ============================================================
print("=" * 60)
print("FLOOR 7: Windows Toast Notifications")
print("=" * 60)

# Q1: Toast notification libraries
toast_hits = grep_files(BROWSER_ROOT,
    r"win10toast|toast|ToastNotification|plyer|windows\.ui\.notifications",
    extensions=[".py", ".js", ".json"])
record("F7-Q1", "Toast notification library references", len(toast_hits) > 0,
       f"{len(toast_hits)} files" if toast_hits
       else "No win10toast or Windows notification references")

# Q2: Push notification / alert system
alert_hits = grep_files(BROWSER_ROOT,
    r"push.?alert|push.?notification|alert.?queue|notification",
    extensions=[".py", ".js"])
record("F7-Q2", "Push notification / alert system", len(alert_hits) > 0,
       f"{len(alert_hits)} files with notification/alert code")

# Q3: YinYang notification bridge
yy_notif = grep_files(BROWSER_ROOT,
    r"yinyang.*notif|push_alerts|delight_engine|alert_queue",
    extensions=[".py", ".js"])
record("F7-Q3", "YinYang OS-level notifications", len(yy_notif) > 0,
       f"{len(yy_notif)} files with YinYang notification code")

# Q4: Actionable notifications
action_hits = grep_files(BROWSER_ROOT,
    r"action.?button|notification.*button|approve.*reject|toast.*action",
    extensions=[".py", ".js"])
record("F7-Q4", "Actionable notification buttons", len(action_hits) > 0,
       f"{len(action_hits)} files" if action_hits
       else "No actionable notification patterns")

In [ ]:
# ============================================================
# Floor 8: Windows Path Handling (backslash vs forward slash)
# Persona: PowerShell Integration Expert (Tower 19, Floor 11)
# Questions probed:
#   Q1: Does Python code use pathlib.Path (OS-agnostic) vs hardcoded paths?
#   Q2: Are there PowerShell scripts for Windows operations?
#   Q3: Does code handle Windows env vars (%APPDATA%, %TEMP%)?
#   Q4: Does the build script detect Windows (MINGW/MSYS)?
# ============================================================
print("=" * 60)
print("FLOOR 8: Windows Path Handling")
print("=" * 60)

# Q1: pathlib usage
pathlib_hits = grep_files(SRC, r"from pathlib import|pathlib\.Path|Path\(",
                          extensions=[".py"])
hardcoded = grep_files(SRC, r'["\']\s*/home/|["\']\s*/tmp/',
                       extensions=[".py"])
record("F8-Q1", "pathlib.Path usage (OS-agnostic)", len(pathlib_hits) > 0,
       f"{len(pathlib_hits)} files use pathlib, {len(hardcoded)} files have hardcoded Unix paths")

# Q2: PowerShell scripts
ps1_files = list(BROWSER_ROOT.rglob("*.ps1"))
ps1_files = [f for f in ps1_files
             if not any(s in str(f) for s in (".git", ".venv", "node_modules"))]
ps1_refs = grep_files(BROWSER_ROOT, r"powershell|pwsh|\.ps1",
                      extensions=[".sh", ".yml", ".yaml", ".json"])
record("F8-Q2", "PowerShell scripts exist", len(ps1_files) > 0 or len(ps1_refs) > 0,
       f"{len(ps1_files)} .ps1 files, {len(ps1_refs)} files reference PowerShell")

# Q3: Windows env vars
win_env = grep_files(BROWSER_ROOT, r"APPDATA|LOCALAPPDATA|USERPROFILE|%TEMP%|os\.environ.*appdata",
                     extensions=[".py", ".js", ".sh", ".ps1"])
record("F8-Q3", "Windows env vars (APPDATA, TEMP)", len(win_env) > 0,
       f"{len(win_env)} files reference Windows env vars" if win_env
       else "No APPDATA/LOCALAPPDATA/USERPROFILE references")

# Q4: Windows platform detection in build scripts
win_detect = bool(re.search(r"MINGW|MSYS|CYGWIN|Windows_NT|uname.*windows", combined_build, re.IGNORECASE))
record("F8-Q4", "Windows platform detection in build", win_detect,
       "MINGW/MSYS/Windows_NT detection found" if win_detect
       else "No Windows platform detection in build scripts")

In [ ]:
# ============================================================
# Floor 9: Windows Service / Scheduled Task
# Persona: Windows Service Expert (Tower 19, Floor 10)
# Questions probed:
#   Q1: Is there a Windows service registration (sc.exe, NSSM)?
#   Q2: Does code reference Task Scheduler (schtasks)?
#   Q3: Is there a daemon/scheduler module that works cross-platform?
#   Q4: Does the build create an auto-start entry?
# ============================================================
print("=" * 60)
print("FLOOR 9: Windows Service / Scheduled Task")
print("=" * 60)

# Q1: Windows service
svc_hits = grep_files(BROWSER_ROOT,
    r"sc\.exe|sc create|nssm|win32service|pywin32|service\.install",
    extensions=[".py", ".ps1", ".sh", ".bat"])
record("F9-Q1", "Windows service registration", len(svc_hits) > 0,
       f"{len(svc_hits)} files" if svc_hits
       else "No Windows service registration code")

# Q2: Task Scheduler
sched_hits = grep_files(BROWSER_ROOT,
    r"schtasks|Task.?Scheduler|taskschd|Register-ScheduledTask",
    extensions=[".py", ".ps1", ".sh", ".bat"])
record("F9-Q2", "Task Scheduler references", len(sched_hits) > 0,
       f"{len(sched_hits)} files" if sched_hits
       else "No schtasks or Task Scheduler references")

# Q3: Cross-platform daemon/scheduler
daemon_hits = grep_files(BROWSER_ROOT,
    r"daemon|scheduler|cron|background.?task|periodic|schedule",
    extensions=[".py"])
record("F9-Q3", "Cross-platform daemon/scheduler module", len(daemon_hits) > 0,
       f"{len(daemon_hits)} Python files with daemon/scheduler code")

# Q4: Auto-start / startup entry
autostart_hits = grep_files(BROWSER_ROOT,
    r"auto.?start|startup|login.?item|HKCU.*Run|CurrentVersion.*Run",
    extensions=[".py", ".ps1", ".sh", ".json"])
record("F9-Q4", "Auto-start / startup entry", len(autostart_hits) > 0,
       f"{len(autostart_hits)} files" if autostart_hits
       else "No auto-start/startup references")

In [ ]:
# ============================================================
# Floor 10: Windows Registry Integration
# Persona: Registry Expert (Tower 19, Floor 6)
# Questions probed:
#   Q1: Does code reference Windows registry (winreg, HKCU, HKLM)?
#   Q2: Are URL protocol handlers registered?
#   Q3: Are file associations (.recipe, .solace) registered?
#   Q4: Is registry cleanup part of the uninstall process?
# ============================================================
print("=" * 60)
print("FLOOR 10: Windows Registry")
print("=" * 60)

# Q1: Registry references
reg_hits = grep_files(BROWSER_ROOT,
    r"winreg|HKCU|HKLM|HKEY_CURRENT_USER|HKEY_LOCAL_MACHINE|Registry",
    extensions=[".py", ".ps1", ".bat", ".iss", ".nsi"])
record("F10-Q1", "Windows registry references", len(reg_hits) > 0,
       f"{len(reg_hits)} files" if reg_hits
       else "No winreg/HKCU/HKLM references")

# Q2: URL protocol handlers
protocol_hits = grep_files(BROWSER_ROOT,
    r"url.?protocol|x-scheme-handler|solace://|RegisterProtocol",
    extensions=[".py", ".js", ".json", ".ps1"])
record("F10-Q2", "URL protocol handler registration", len(protocol_hits) > 0,
       f"{len(protocol_hits)} files" if protocol_hits
       else "No URL protocol handler registration")

# Q3: File associations
assoc_hits = grep_files(BROWSER_ROOT,
    r"file.?association|ftype|assoc|\.recipe|\.solace|mime.?type",
    extensions=[".py", ".json", ".ps1", ".iss"])
record("F10-Q3", "File association registration", len(assoc_hits) > 0,
       f"{len(assoc_hits)} files" if assoc_hits
       else "No file association references")

# Q4: Uninstall / registry cleanup
uninstall_hits = grep_files(BROWSER_ROOT,
    r"uninstall|remove.*registry|cleanup|UninstallDelete",
    extensions=[".py", ".ps1", ".iss", ".nsi", ".sh"])
record("F10-Q4", "Uninstall / registry cleanup", len(uninstall_hits) > 0,
       f"{len(uninstall_hits)} files" if uninstall_hits
       else "No uninstall or registry cleanup references")

In [ ]:
# ============================================================
# EVIDENCE SUMMARY -- Tower 19: Windows QA x Solace Browser
# ============================================================

passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
pass_rate = (passed / total * 100) if total > 0 else 0

floor_summary = {}
for r in results:
    floor = r["id"].split("-")[0]
    if floor not in floor_summary:
        floor_summary[floor] = {"passed": 0, "failed": 0, "total": 0}
    floor_summary[floor]["total"] += 1
    if r["status"] == "PASS":
        floor_summary[floor]["passed"] += 1
    else:
        floor_summary[floor]["failed"] += 1

floor_labels = {
    "F1": "MSI/NSIS Installer", "F2": "Defender / SmartScreen",
    "F3": "Fluent Design / UI", "F4": "High DPI Scaling",
    "F5": "Credential Storage", "F6": "Taskbar / Tray",
    "F7": "Toast Notifications", "F8": "Path Handling",
    "F9": "Service / Sched Task", "F10": "Registry",
}

summary = {
    "surface": NORTHSTAR,
    "tower": TOWER,
    "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total,
    "passed": passed,
    "failed": failed,
    "pass_rate_pct": round(pass_rate, 1),
    "verdict": "STRONG" if pass_rate >= 70 else "PARTIAL" if pass_rate >= 40 else "WEAK",
    "floors": {}
}

print(f"\n{'='*60}")
print(f"TOWER 19: Windows QA -- EVIDENCE SUMMARY")
print(f"{'='*60}")
print(f"Total probes: {total}")
print(f"PASS: {passed} ({pass_rate:.1f}%)")
print(f"FAIL: {failed}")
print(f"Verdict: {summary['verdict']}")
print(f"\n{'Floor':<6} {'Label':<25} {'Pass':<6} {'Fail':<6} {'Total':<6}")
print("-" * 55)
for fid in sorted(floor_summary.keys(), key=lambda x: int(x[1:])):
    fs = floor_summary[fid]
    label = floor_labels.get(fid, "Unknown")
    print(f"{fid:<6} {label:<25} {fs['passed']:<6} {fs['failed']:<6} {fs['total']:<6}")
    summary["floors"][fid] = {"label": label, **fs}

print(f"\n{'='*60}")
print("Evidence JSON:")
print(json.dumps(summary, indent=2))